![](https://storage.googleapis.com/kaggle-competitions/kaggle/29548/logos/header.png?t=2021-06-16-09-31-18)

## **Final project: Example solution**
-----

Welcome to the example solution for the final task of the "Machine Learning: Theory and Application" course. 
It was developed for students, who took part in **Peter the Great St.Petersburg Polytechnic University Summer school**. In spite of this, if you wasn't on the course, i put this competition for free with the public test dataset. You are wellcome to share you solution and put your comments to emproove this competition

This notebook - is just an example solution, which i've created for students to get started. Do not treat it as the best solution, it is not the goal of this notebook. I especially have used some simple concepst in modeling part, to give you more wide field for your creativity. My goal is to show you some main steps of how to work with data to create ML models. Those basic steps are:
1. Explorational Data Analysis 
2. Data Preparation
3. Feature engeneering
4. ML model selection

At first, lets make a few important preparations:


In [ ]:
import pandas as pd
import seaborn as sns
import plotly.express as xp
import plotly.graph_objects as go
import numpy as np
import numpy as np
from datetime import datetime
import missingno
import yaml
from collections import Counter
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV,ShuffleSplit
from sklearn.manifold import TSNE
from sklearn.linear_model import RidgeClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier, ExtraTreesClassifier, RandomForestClassifier

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

palette = ['#3aa833',"#6b856a","#354014"]
sns.palplot(palette)

Some methods, which will help us to work with values, separated with "|". Use them in case you do not know how to process such values. Also, other ideas are welcome.

In [ ]:
def split_to_onehot(df, col):
    """
    This method converts features separated by '|' into one-hot vectors.
    Additionally it drops unnecessary values, which present only in 
    test set / train set or have only one value.
    """
    # Getting all unique ganres values.
    unique = []
    for i in df.index:
        unique.extend(df.loc[i,col].split("|"))
    if "" in unique:
        unique.remove("")
    unique = list(set(unique))
    
    # Putting values into binary form 
    onehot = df.loc[:,["Category"]]
    onehot[unique] = np.zeros((len(unique),), dtype = np.int8)
    for i in df.index:
        g = set(df.loc[i,col].split("|"))
        for j in g:
            if j!="":
                onehot.loc[i,j] = 1
                
    # Dropping unnecessary values            
    _a = onehot.groupby("Category").sum()
    only_one = list(_a.sum()[_a.sum()==1].index)
    only_train = list(_a.loc["none"][_a.loc["none"]==0].index)
    only_test = list(_a.loc[["like",'dislike']].sum()[_a.loc[["like",'dislike']].sum()==0].index)
    _a = set(only_one + only_train + only_test)
    onehot = onehot.drop(_a, axis=1)
    
    return onehot

def onehot_to_tsne2(df, title):
    """
    This method converts one-hot representation into two tsne values.
    Such operation is needed to shrink the dimentionality of the dataset
    """
    onehot = df.drop("Category",axis=1)
    embedding = TSNE(n_components=2, init="pca")
    embedded = embedding.fit_transform(onehot)
    embedded = pd.DataFrame(embedded,columns=[f"{title}_tsne1",f"{title}_tsne2"])
    return embedded

def plot_commulative_onehot(onehot):
    """
    Method of plotting commulative values of the one hot feature representation
    """
    _df = onehot.groupby("Category").sum()
    fig = go.Figure()
    for i in range(len(_df.index)):
        k = _df.index[i]
        x,y=[],[]
        for g in _df.columns:
            if _df.loc[k,g]!=0:
                x.append(g)
                y.append(_df.loc[k,g])
        fig.add_trace(go.Bar(x=x, y=y,name=k,marker=dict(color=palette[i])))
    fig.show()

Before we start we need to download the train/test sets. To simplify the operations with the dataset and its visualisation - we will put all the data togather and create the mask, to separate train and test sets.

In [ ]:
PATH = "../input/mymusicalprefrences/" 
train = pd.read_csv(f"{PATH}train.csv")
test = pd.read_csv(f"{PATH}test.csv")
description = yaml.load(open(f"{PATH}Description.yaml",'r'),Loader=yaml.FullLoader)
df = pd.concat([train,test]).reset_index(drop=True)
tr_mask = ~df.Category.isna()

# 1. Explorational Data Analysis 

On this part we analise the data. We have to check what kind of values do we have in it. How many categorical and continuouse attributes do we have. If there are any None values and what can we do with it.

In [ ]:
df.columns = [i.strip() for i in df.columns]
print(set(df.columns))

In [ ]:
df.describe()

In [ ]:
# Displaing of None values in the dataset
missingno.bar(df, color=palette, figsize=(30,2))

In [ ]:
cat_features = {"Artists","Track","Version","Artists_Genres","Album","Album_type","Labels","Vocal","Country","Key"}
con_features = {"Duration","Release_year","BPM","Energy","Dancebility","Happiness"}
display(df[cat_features].head())
display(df[con_features].head())

In [ ]:
sns.pairplot(df[list(con_features)+["Category"]],palette=palette[:2], hue="Category")

In [ ]:
# For more easy usage of the Category feature
df["Category"] = df["Category"].fillna("none").replace({0:"dislike",1:"like"})

# 2. Data Preparation and Feature Engeneering

In this part we will analyze data attributes, and transform them to the for strict form. During this process we will analize the features and find some supposed dependancies in the data. For that cases, that may be transformed - we will create new features, which will describe our data better. I the end of this step we will get the dataset, ready for modeling with ML algorythms.

## 2.1 Key feature

In [ ]:
description["Key"]

In [ ]:
xp.scatter(df, x="Key", y="Track",color="Category", height=500, color_discrete_sequence=palette)

In [ ]:
# We correct strings and replace some ambivalent values
df["isMajor"], df["Key"] = df["Key"].apply(lambda x: x.split(" ")[1]), df["Key"].apply(lambda x: x.split(" ")[0])
df.loc[:,"Key"] = df["Key"].replace({"D♭": "C#", "E♭": "D#", "G♭": "F#", "A♭": "G#","B♭":"A#"})
xp.scatter(df, x="Key", y="Track",color="Category", height=500, color_discrete_sequence=palette)

In [ ]:
# We put the Major/Minor part into new feature, to make it more easy for our model to fit on it
df.loc[:,"isMajor"] = (df["isMajor"]=="Major").astype(int)
_df = df.groupby(["isMajor","Category"], as_index=False).count()
xp.bar(_df,x="isMajor", y="Track",color="Category", height=400, color_discrete_sequence=palette)

In [ ]:
_df = df.copy(deep=True)
_df["Key_percise"] = _df["Key"] +"_major:"+ _df["isMajor"].astype(str)
_df = _df.groupby(["Key_percise","Category"], as_index=False).count()
xp.bar(_df, x="Key_percise", y="Track", color="Category", height=500, color_discrete_sequence=palette)

From the analisys of this feature we can say:
* Major Tracks has more "likes" then Minor ones - **190/142=1.34** vs **161/172=0.94**
* D# looks like the most disliked Key but at the same time - there are not much tracks on this key, may not be much significant.
* The biggest proportion of likes/dislikes is in A Major key - **11/4 = 2.75** and C Major key - **36/17 = 2.117**

In [ ]:
df[list(set(df["Key"].values))] = OneHotEncoder().fit_transform(df[["Key"]]).toarray()
df = df.drop("Key", axis=1)

## 2.2 Release year feature

In [ ]:
description["Release year"]

In [ ]:
xp.scatter(df, x="Release_year", y="Track",color="Category", height=500, color_discrete_sequence=palette)

In [ ]:
# Lets create the decade feature, to detect some music of 80th, 90th etc. as a specific janre
df.loc[:,"Release_decade"] = (df.loc[:,"Release_year"]//10 * 10)
# Cause of the small number of values, we will put all <90th toone value, called 80th
df.loc[df.loc[:,"Release_decade"]<1990,"Release_decade"] = 1980 
_df = df.groupby(["Release_decade","Category"], as_index=False).count()
xp.bar(_df,x="Release_decade", y="Track",color="Category",height=500, color_discrete_sequence=palette)

From the analisys of this feature we can get the porportion of likes/ dislikes:

| Decade | like/dislike  | koeff |
| ------ |:-------------:| -----:|
| 1980s  | 9/6           | 1.5   |
| 1990s  | 16/8          | 2     |
| 2000s  | 81/90         | 0.9   |
| 2010s  | 213/153       | 1.39  |
| 2020s  | 32/57         | 0.56  |

Number of values in decades less then 1990 - may not be destinctive, but according to the koefficient - this music i like the most (this is true).

At the same time, the modern music (Like, two last years) is not in my top .

## 2.3 Genres features

In [ ]:
description["Artists Genres"]

In [ ]:
ganres_onehot = split_to_onehot(df, "Artists_Genres")
plot_commulative_onehot(ganres_onehot)

From the analisys of this feature we can say:
* There are some ganres that I do not like (or not really): **dance, house, dnb, latinfolk, epicmetal, electronics, ruspop**
* There are some ganres that I do mostly like: **rock, indie, pop, rnd, soul, numetal**

We can see, that we have to much values in one-hot vector representation. It may cause much problems, during the fitting of our model (it couldn't find distinctive dependencies in sparse matrix). In this case we will apply TSNE as a handy solution, to transfer our n columns into two, without losing the information. I recommend you to experiment with this step during your work.

In [ ]:
genres_embedded = onehot_to_tsne2(ganres_onehot, "Genres")
_df = genres_embedded.copy(deep=True)
_df[["Category","Artists_Genres"]] = df[["Category","Artists_Genres"]]
xp.scatter(_df,x="Genres_tsne1",y="Genres_tsne2",color="Category", hover_data=["Artists_Genres"], height=500, color_discrete_sequence=palette)

In [ ]:
df = pd.concat([df,genres_embedded], axis=1)
df = df.drop("Artists_Genres", axis=1)

## 2.4 Energy,Happiness,Dancebility, BPM

In [ ]:
for k in ["Energy","Happiness","Dancebility","BPM"]:
    print(f"{k}:{description[k]}")

In [ ]:
df["BPM"] = df["BPM"].apply(lambda x: str(x)[1:] if str(x)[0]=='`' else x)
df[['Energy', 'Happiness', 'Dancebility','BPM']] = df[['Energy', 'Happiness', 'Dancebility','BPM']].fillna(0)
df[['Energy%', 'Happiness%', 'Dancebility%']] = df[['Energy', 'Happiness', 'Dancebility']].apply(lambda x: x/sum(x), axis=1)
df[['Energy%', 'Happiness%', 'Dancebility%']] = df[['Energy%', 'Happiness%', 'Dancebility%']].fillna(0)

Nothing to do here. We will just add some propotional values of the same features.

## 2.5 Labels

In [ ]:
print(description["Labels"])

In [ ]:
df.Labels = df.Labels.fillna('NA')
labels_onehot = split_to_onehot(df, "Labels")
plot_commulative_onehot(labels_onehot)

In [ ]:
labels_embedded = onehot_to_tsne2(labels_onehot, "Labels")
_df = labels_embedded.copy(deep=True)
_df[["Category","Labels"]] = df[["Category","Labels"]]
xp.scatter(_df,x="Labels_tsne1",y="Labels_tsne2",color="Category", hover_data=["Labels"], height=500, color_discrete_sequence=palette)

In [ ]:
df = pd.concat([df,labels_embedded[["Labels_tsne1","Labels_tsne2"]]], axis=1)
df = df.drop("Labels", axis=1)

## 2.6 Artists

This feature is the most complicated one. For the example, lets assume, that the main artist for the track - named the 1st. Others will be named as collaborators.This is cause for not to make the one-hot matrix too spars putting all the artists to the single columns. Experiment with this feature - the result of its preprocessing may influence significantly to the final result.

In [ ]:
print(description["Artists"])

In [ ]:
df.Artists = df.Artists.fillna("NA")
allstars = []
for i in df.index:
    allstars.extend(df.loc[i, "Artists"].split("|"))
len(set(allstars))

In [ ]:
# We will put some threshold, not to put some rare artists into one-hot vector.
threshold = 3
others = Counter(allstars)
others = [k for k in others if others[k]<=threshold]
len(others)

In [ ]:
# Drop all artists who are just in test set or just in train set
in_train, in_test = [], []
for i in df.loc[tr_mask].index:
    in_train.extend(df.loc[i, "Artists"].split("|"))
for i in df.loc[~tr_mask].index:
    in_test.extend(df.loc[i, "Artists"].split("|"))
    
only_test = set(in_test) - set(in_train)
only_train = set(in_train) - set(in_test)
display(len(only_test))
display(len(only_train))

In [ ]:
allstars = list(set(allstars) - set(others) - only_test - only_train)
print(len(allstars))
others = set(others) | only_test | only_train
print(len(others))

In [ ]:
res = []
def prune(x):
    vector = np.zeros(len(allstars)+1) #for others
    x = [i for i in x.split("|")]
    for i in range(len(allstars)):
        vector[i]=1 if allstars[i] in x else 0
    if len(x)>sum(vector):
        vector[-1]=1
    res.append(vector)

df["Artists"].apply(prune)
onehot_artists = pd.DataFrame(res, columns = allstars+["Others"], index=df.index)

In [ ]:
onehot_artists

In [ ]:
df["Other_Artists"] = onehot_artists["Others"]
onehot_artists = onehot_artists.drop("Others", axis=1)
onehot_artists["Category"] = df["Category"]

In [ ]:
plot_commulative_onehot(onehot_artists)

From the analisys of this feature we can say:
* There are some favorite artists, such as: **twenty one pilots, radiohead, monatik, rhcp, скриптонит, etc.**
* There are some artists that i do not like much: **morgenshtern, 6ix9ine, Элджей, Lady GaGa, Pitbul etc.**

In [ ]:
artists_embedded = onehot_to_tsne2(onehot_artists, "Artists")
_df = artists_embedded.copy(deep=True)
_df[["Category","Artists"]] = df[["Category","Artists"]]
xp.scatter(_df,x="Artists_tsne1",y="Artists_tsne2",color="Category", hover_data=["Artists"], height=500, color_discrete_sequence=palette)

In [ ]:
df = pd.concat([df,artists_embedded[["Artists_tsne1","Artists_tsne2"]]], axis=1)
df = df.drop("Artists", axis=1)

## 2.7 Tracks, Version, Album_type

In [ ]:
for i in ["Track", "Version", "Album_type"]:
    print(description[i])

In this cases we do not have much features (so our one-hot representation wont be too spars) so we will apply onehot or label encoders (dependong of the number of features) to encode out values

### 2.7.1 Tracks

In [ ]:
artists_encoder = LabelEncoder()
df["Track"] = artists_encoder.fit_transform(df["Track"])

### 2.7.2 Version

In [ ]:
_df = df.groupby(["Version","Category"], as_index=False).count()
xp.bar(_df,x="Version",y="Id",color="Category", color_discrete_sequence=palette)

In [ ]:
df["Version"] = df["Version"].fillna("NA")
versions = set(df["Version"])
df[list(versions)] = OneHotEncoder().fit_transform(df[["Version"]]).toarray()
df = df.drop(["Version","NA"], axis=1)

### 2.7.3 Album_type

In [ ]:
_df = df.groupby(["Album_type","Category"], as_index=False).count()
xp.bar(_df,x="Album_type",y="Id",color="Category", color_discrete_sequence=palette)

In [ ]:
df["Album_type"] = df["Album_type"].fillna("NA")
versions = set(df["Album_type"])
df[list(versions)] = OneHotEncoder().fit_transform(df[["Album_type"]]).toarray()
df = df.drop(["Album_type","NA"], axis=1)

## 2.8 Album

In [ ]:
print(description["Album"])

In [ ]:
df["Album"] = df["Album"].fillna("NA")
ganres_onehot = split_to_onehot(df, "Album")
plot_commulative_onehot(ganres_onehot)

In [ ]:
album_embedded = onehot_to_tsne2(onehot_artists, "Album")
_df = album_embedded.copy(deep=True)
_df[["Category","Album"]] = df[["Category","Album"]]
xp.scatter(_df,x="Album_tsne1",y="Album_tsne2",color="Category", hover_data=["Album"], height=500, color_discrete_sequence=palette)

In [ ]:
df = pd.concat([df,album_embedded[["Album_tsne1","Album_tsne2"]]], axis=1)
df = df.drop("Album", axis=1)

## 2.8 Vocal

In [ ]:
print(description["Vocal"])

In [ ]:
df["Vocal"] = df["Vocal"].fillna('N')
onehot = np.zeros((len(df),2))
for i in range(len(df)):
    v = df.iloc[i]["Vocal"]
    if v == 'F':
        onehot[i] = [1,0]
    elif v == 'M':
        onehot[i] = [0,1]
    elif v == 'F|M':
        onehot[i] = [1,1]
df[["Fem_voc","Mal_voc"]] = onehot
df = df.drop("Vocal",axis=1)

## 2.9 Country

In [ ]:
print(description["Country"])

In [ ]:
df["Country"] = df["Country"].fillna("NA")
country_onehot = split_to_onehot(df, "Country")
plot_commulative_onehot(country_onehot)

In [ ]:
country_onehot = country_onehot.drop("Category", axis=1)
df = pd.concat([df,country_onehot], axis=1)
df = df.drop("Country", axis=1)

## 3 Model selection

In this part I represent the simple example of model selection. 

RidgeClassifierWe will take one specific algorythm (RidgeClassifier - to show the concept of hyperrparameter tuning) and try to find the best configuration of the model. At the same time we will apply shuffle split to validate our model. The best result, should be the best solution (but usually may not be the best one on the test set). Experiment with the different technologies, libraries and approaches, to achieve the better result.

In [ ]:
x, y = df.loc[tr_mask].iloc[:,2:], df.loc[tr_mask,"Category"]
deploy = df.loc[~tr_mask].iloc[:,2:]

### 3.1 CatBoost classifier

In [ ]:
model_c = CatBoostClassifier(verbose=False)
grid = {"l2_leaf_reg" : [3,4,5,6,7,8,9,10],
        "random_strength" : [0.9],
        "learning_rate" : [0.00855],
        "depth" : [6]}
cv = ShuffleSplit(n_splits=5,random_state=0) 
clf = GridSearchCV(model_c, grid, cv=cv)
clf.fit(x,y)

In [ ]:
pd.DataFrame(clf.cv_results_).sort_values("rank_test_score")["params"].values

In [ ]:
model_c=clf.best_estimator_
sample = pd.read_csv(f"{PATH}sample_submition.csv")
sample["Category"] = model_c.predict(deploy)
sample["Category"] = (sample["Category"]=="like").astype(int)

In [ ]:
sample.to_csv("deploy_cat.csv", index=False)

### 3.2 SVM classifier

In [ ]:
model_s = SVC()
grid = {"kernel" : ['poly', 'rbf','linear'],
        "C" : [.1,1,10]}
cv = ShuffleSplit(n_splits=5,random_state=0) 
clf = GridSearchCV(model_s, grid, cv=cv)
clf.fit(x,y)

In [ ]:
pd.DataFrame(clf.cv_results_).sort_values("rank_test_score")["params"].values

In [ ]:
model_s=clf.best_estimator_
sample = pd.read_csv(f"{PATH}sample_submition.csv")
sample["Category"] = model_s.predict(deploy)
sample["Category"] = (sample["Category"]=="like").astype(int)

In [ ]:
sample.to_csv("svm.csv", index=False)

### 3.3 Logistic Regression

In [ ]:
model_l = LogisticRegression(penalty='elasticnet',solver='saga',max_iter=9000)
grid = {"l1_ratio" : [0,.05,.1,]}
cv = ShuffleSplit(n_splits=5,random_state=0) 
clf = GridSearchCV(model_l, grid, cv=cv)
clf.fit(x,y)

In [ ]:
pd.DataFrame(clf.cv_results_).sort_values("rank_test_score")["params"].values

In [ ]:
model_l=clf.best_estimator_
sample = pd.read_csv(f"{PATH}sample_submition.csv")
sample["Category"] = model_l.predict(deploy)
sample["Category"] = (sample["Category"]=="like").astype(int)

In [ ]:
sample.to_csv("linear.csv", index=False)

### 3.4 Voting Classifier for best parameters

In [ ]:
model = VotingClassifier(voting='hard',estimators=[
    ('cat', model_c), 
    ('svm', model_s), 
    ('lin', model_l), 
    ('extra', ExtraTreesClassifier()),
    ('rf',RandomForestClassifier())
])
model.fit(x,y)

In [ ]:
sample = pd.read_csv(f"{PATH}sample_submition.csv")
sample["Category"] = model.predict(deploy)
sample["Category"] = (sample["Category"]=="like").astype(int)

In [ ]:
sample.to_csv("voting.csv", index=False)